In [0]:
#Silver invoices, pulling fields out of free text
#Parse the invoices
from pyspark.sql import functions as F

CATALOG = "capstone_supply_chain"

df_inv = spark.table(f"{CATALOG}.bronze.raw_invoices")
txt = F.col("raw_content")

def extract(pattern):
    """Return the text matched by the (group), or NULL if nothing matched."""
    val = F.regexp_extract(txt, pattern, 1)
    return F.when(val == "", F.lit(None).cast("string")).otherwise(F.trim(val))

df_parsed = (df_inv
    .withColumn("invoice_number",   extract(r"INVOICE(?: NUMBER)?:\s*([\w-]+)"))
    .withColumn("invoice_date_str", extract(r"(?<!DUE )\bDATE:\s*(\w+ \d{1,2},\s*\d{4})"))
    .withColumn("po_reference",     extract(r"(PO-\d{4}-\d+)"))
    .withColumn("vendor_name",      extract(r"(?:FROM|SELLER):\s*([^,\n]+)"))
    .withColumn("total_amount_str", extract(r"(?m)^[ \t]*TOTAL[^:\n]*:\s*(?:USD|EUR|CAD)?\s*[$€]?\s*([\d,]+(?:\.\d+)?)"))
    .withColumn("currency",         extract(r"(?m)^[ \t]*TOTAL[^\n]*?\b(USD|EUR|CAD)\b"))
    .withColumn("payment_terms",    extract(r"\b(Net \d+)\b"))
    .withColumn("invoice_date",
        F.expr("CAST(try_to_timestamp(invoice_date_str, 'MMMM d, yyyy') AS DATE)"))
    .withColumn("total_amount",
        F.expr("try_cast(replace(total_amount_str, ',', '') AS DOUBLE)"))
    .select(
        "invoice_number", "invoice_date", "po_reference", "vendor_name",
        "total_amount", "currency", "payment_terms",
        "filename",
        F.col("content_length").alias("source_doc_length"),
        F.current_timestamp().alias("silver_processed_at"),
    )
)

df_parsed.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.invoices_parsed")
print("✅ silver.invoices_parsed rows:", df_parsed.count())
display(df_parsed)

In [0]:
#Check that nothing came out empty
df = spark.table(f"{CATALOG}.silver.invoices_parsed")

# 1. Null check (we want all zeros)
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in ["invoice_number", "invoice_date", "po_reference",
              "vendor_name", "total_amount", "currency", "payment_terms"]
]).show()

# 2. Look at the VALUES, not just the nulls
df.select("invoice_number", "invoice_date", "po_reference", "vendor_name",
          "total_amount", "currency", "payment_terms").show(truncate=False)

In [0]:
#If a field is NULL for invoice 2
print(spark.table(f"{CATALOG}.bronze.raw_invoices")
      .where("filename = 'sample_invoice_2.txt'").first()["raw_content"])

In [0]:
#bonus check
from pyspark.sql import functions as F

po = spark.table("capstone_supply_chain.silver.purchase_orders")
inv = spark.table("capstone_supply_chain.silver.invoices_parsed")

(inv.join(po, inv["po_reference"] == po["po_id"], "left")
    .select(
        inv["invoice_number"],
        inv["po_reference"],
        inv["vendor_name"].alias("invoice_vendor"),
        po["vendor_name"].alias("po_vendor"),
        inv["total_amount"].alias("invoice_total"),
        po["line_total"].alias("po_line_total"),
        inv["currency"].alias("invoice_currency"),
        po["currency"].alias("po_currency"),
    )
    .show(truncate=False))